## LLM Consistency Testing with DeMorgan's Law Mutations

This notebook contains code for testing code inconsistency using DeMorgan's law mutations on boolean expressions

In [1]:
import os
import sys
import ast
import pandas as pd

In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from llm_models.code_llms import Mistral
from code_inconsistency.code_inconsistency_tester import LLMConsistencyTester
from code_inconsistency.prompt_templates.prompt_template import CodeInconsistencyPromptTemplate

In [4]:
class DeMorganPreConditionFilter:
    """
    Helper class to filter database entries that contain boolean operations 
    suitable for DeMorgan's law transformations.
    """
    
    @staticmethod
    def has_boolean_operations(code_str: str) -> bool:
        """
        Check if the code contains boolean operations (and/or) that can be mutated with DeMorgan's laws.
        
        Args:
            code_str (str): The code to analyze
            
        Returns:
            bool: True if code contains boolean operations suitable for DeMorgan mutation
        """
        try:
            tree = ast.parse(code_str)
            
            # Walk through all nodes to find boolean operations
            for node in ast.walk(tree):
                # Check for BoolOp nodes (and/or operations)
                if isinstance(node, ast.BoolOp):
                    if isinstance(node.op, (ast.And, ast.Or)):
                        return True
                
                # Check for UnaryOp with Not that could be applied to boolean operations
                if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.Not):
                    if isinstance(node.operand, ast.BoolOp):
                        return True
                
            return False
            
        except (SyntaxError, ValueError):
            # If code can't be parsed, assume it's not suitable for mutation
            return False
    
    @staticmethod
    def filter_database_for_demorgan(llmtester):
        """
        Filter the database to only include entries that can be mutated with DeMorgan's laws.
        
        Args:
            llmtester: LLMConsistencyTester instance
            
        Returns:
            list: List of filtered document IDs suitable for DeMorgan mutation
        """
        suitable_docs = []
        
        # Get all documents from the database
        all_docs = list(llmtester.question_database.find({}))
        
        print(f"Checking {len(all_docs)} documents for DeMorgan mutation suitability...")
        
        for doc in all_docs:
            # Check if the solution contains boolean operations
            if 'full_sol' in doc and DeMorganPreConditionFilter.has_boolean_operations(doc['full_sol']):
                suitable_docs.append(doc['_id'])
        
        print(f"Found {len(suitable_docs)} documents suitable for DeMorgan mutation out of {len(all_docs)} total documents")
        
        return suitable_docs

In [5]:
llmtester = LLMConsistencyTester("HumanEval_Input_Output")

# Test basic database connection and row loading
print(f"Database connection established")
total_docs = llmtester.question_database.count_documents({})
print(f"Total documents in database: {total_docs}")

# Show a sample document to verify structure
sample_doc = llmtester.question_database.find_one({})
if sample_doc:
    print(f"\nSample document structure:")
    print(f"Document ID: {sample_doc['_id']}")
    print(f"Has 'full_sol' field: {'full_sol' in sample_doc}")
    if 'full_sol' in sample_doc:
        print(f"Solution preview: {sample_doc['full_sol'][:100]}...")
else:
    print("No documents found in database")

MongoDB connected
Database connection established
Total documents in database: 1111

Sample document structure:
Document ID: HumanEvalTF0
Has 'full_sol' field: True
Solution preview: from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
   ...


In [6]:
# Filter documents that can be mutated with DeMorgan's laws
demorgan_suitable_docs = DeMorganPreConditionFilter.filter_database_for_demorgan(llmtester)
print(f"\nDocuments suitable for DeMorgan mutation: {len(demorgan_suitable_docs)}")

Checking 1111 documents for DeMorgan mutation suitability...
Found 257 documents suitable for DeMorgan mutation out of 1111 total documents

Documents suitable for DeMorgan mutation: 257


In [7]:
llm = Mistral()

In [8]:
model_name = "mistral-small-2506"
mistral_results = os.path.join(proj_dir + '/results/code_inconsistencies/mistral')
os.makedirs(mistral_results, exist_ok=True)

In [9]:
print(f"\nDocuments suitable for DeMorgan mutation: {len(demorgan_suitable_docs)}")
print(demorgan_suitable_docs)


Documents suitable for DeMorgan mutation: 257
['HumanEvalTF144', 'HumanEvalTF145', 'HumanEvalTF146', 'HumanEvalTF147', 'HumanEvalTF148', 'HumanEvalTF149', 'HumanEvalTF150', 'HumanEvalTF151', 'HumanEvalTF260', 'HumanEvalTF261', 'HumanEvalTF262', 'HumanEvalTF263', 'HumanEvalTF264', 'HumanEvalTF265', 'HumanEvalTF266', 'HumanEvalTF267', 'HumanEvalTF272', 'HumanEvalTF273', 'HumanEvalTF274', 'HumanEvalTF275', 'HumanEvalTF276', 'HumanEvalTF306', 'HumanEvalTF307', 'HumanEvalTF308', 'HumanEvalTF309', 'HumanEvalTF310', 'HumanEvalTF311', 'HumanEvalTF312', 'HumanEvalTF375', 'HumanEvalTF376', 'HumanEvalTF377', 'HumanEvalTF378', 'HumanEvalTF379', 'HumanEvalTF380', 'HumanEvalTF381', 'HumanEvalTF382', 'HumanEvalTF383', 'HumanEvalTF440', 'HumanEvalTF441', 'HumanEvalTF442', 'HumanEvalTF443', 'HumanEvalTF444', 'HumanEvalTF445', 'HumanEvalTF446', 'HumanEvalTF447', 'HumanEvalTF454', 'HumanEvalTF455', 'HumanEvalTF456', 'HumanEvalTF457', 'HumanEvalTF458', 'HumanEvalTF459', 'HumanEvalTF460', 'HumanEvalTF461'

In [14]:
# Run No-Mutation consistency test on filtered documents (Commented  since tests are done and results are saved)

if len(demorgan_suitable_docs) > 0:
   # Set up dynamic file naming like the original tester
   syntactic_mutation = None
   prompt_type = "zero_shot"
   mutation_str = f"{syntactic_mutation}_mutation"
  
   output_file_path = f"{mistral_results}/{model_name}_{prompt_type}_{mutation_str}.csv"
   print(f"Results will be saved to: {output_file_path}")

    # MODIFICATION: Test only first 10 documents
   test_docs = demorgan_suitable_docs[10:20]
   print(f"Running No Mutation consistency test on {len(test_docs)} suitable documents")
   print('llmtester: ', llmtester)
   print(f"Starting processing of {len(test_docs)} documents...")
   print()
   
   # Use the unified run_code_consistency_test method with DeMorgan mutation
   pass_count = llmtester.run_code_consistency_test(
       prompt_helper=CodeInconsistencyPromptTemplate.OutputPrediction.zero_shot_prompt,
       prompt_type=prompt_type,
       output_file_path=output_file_path,
       specific_doc_ids=test_docs,
       syntactic_mutation=syntactic_mutation,
       task_set="HumanEval",
       num_tests=llmtester.question_database.count_documents({})
   )
   
   print(f"\nNo Mutation consistency test completed!")
   print(f"Processed: {len(test_docs)} documents")
   print(f"Pass count: {pass_count}")
   print(f"Results saved to: {output_file_path}")
else:
   print("No documents found suitable for DeMorgan mutation. Skipping test.")

Results will be saved to: /Users/jin/Documents/GitHub/Code Reasoning Model Research Project/results/code_inconsistencies/mistral/mistral-small-2506_zero_shot_None_mutation.csv
Running No Mutation consistency test on 10 suitable documents
llmtester:  <code_inconsistency.code_inconsistency_tester.LLMConsistencyTester object at 0x12a92ad10>
Starting processing of 10 documents...

Testing 10 specific documents


  1%|          | 11/1111 [00:43<1:11:42,  3.91s/it]

HumanEvalTF11

No Mutation consistency test completed!
Processed: 10 documents
Pass count: 0
Results saved to: /Users/jin/Documents/GitHub/Code Reasoning Model Research Project/results/code_inconsistencies/mistral/mistral-small-2506_zero_shot_None_mutation.csv


In [ ]:
# Run DeMorgan mutation consistency test on filtered documents
if len(demorgan_suitable_docs) > 0:
    # Set up dynamic file naming like the original tester
    syntactic_mutation = "demorgan"
    prompt_type = "zero_shot"
    mutation_str = f"{syntactic_mutation}_mutation"
    
    output_file_path = f"{mistral_results}/{model_name}_{prompt_type}_{mutation_str}.csv"
    print(f"Results will be saved to: {output_file_path}")
    
    # MODIFICATION: Test only first 5 documents
    test_docs = demorgan_suitable_docs[10:20]
    print(f"Running DeMorgan consistency test on {len(test_docs)} suitable documents")

    print('llmtester: ', llmtester)

    print()
    
    # Use the unified run_code_consistency_test method with DeMorgan mutation
    pass_count = llmtester._run_code_consistency_test(
        llm=llm,
        prompt_helper=CodeInconsistencyPromptTemplate.OutputPrediction.zero_shot_prompt,
        prompt_type=prompt_type,
        output_file_path=output_file_path,
        specific_doc_ids=test_docs,
        syntactic_mutation=syntactic_mutation
    )
    
    print(f"\nDeMorgan consistency test completed!")
    print(f"Results saved to: {output_file_path}")
else:
    print("No documents found suitable for DeMorgan mutation. Skipping test.")

## Analysis

This notebook tests LLM consistency using DeMorgan's law mutations, which transform boolean expressions:
- `A and B` → `not ((not A) or (not B))`
- `A or B` → `not ((not A) and (not B))`
- `not (A and B)` → `(not A) or (not B)`
- `not (A or B)` → `(not A) and (not B)`

The pre-condition filter ensures we only test on code that contains boolean operations suitable for these transformations.